# ECE 227 — Network Analysis Runner

Run all cells **top-to-bottom**.
The final cell exports every figure to **`network_analysis_results.pdf`**.

| Section | Dataset | Analysis |
|---------|---------|----------|
| 1 | Data download | Higgs Twitter edgelists |
| 2 | Higgs Mention | Degree dist · Centrality · Intersections |
| 3 | Higgs Reply | Degree dist · Centrality · Intersections |
| 4 | Higgs Retweet | Degree dist · Centrality · Intersections |
| 5 | Cross-network | Comparison & overlap |
| 6 | Enron Email | Degree dist · Centrality · Communities |
| 7 | Facebook | Degree dist · Communities |
| 8 | PDF Export | All figures → `network_analysis_results.pdf` |


In [ ]:
import os, json, warnings
from collections import Counter

import numpy as np
import networkx as nx
import igraph as ig
import matplotlib.pyplot as plt
from matplotlib.backends.backend_pdf import PdfPages

%matplotlib inline
warnings.filterwarnings('ignore')
plt.rcParams.update({
    'figure.dpi': 100,
    'axes.titlesize': 12,
    'axes.labelsize': 10,
    'figure.autolayout': True,
})

ROOT       = os.getcwd()
DATA_DIR   = os.path.join(ROOT, 'data')
PAPER_FIGS = os.path.join(ROOT, 'Paper', 'figures')
os.makedirs(PAPER_FIGS, exist_ok=True)
all_figures = []   # (fig) — appended by every analysis section

def savefig(fig, name):
    fig.savefig(os.path.join(PAPER_FIGS, name), dpi=150, bbox_inches='tight')

print(f"Root       : {ROOT}")
print(f"Paper figs : {PAPER_FIGS}")
print("Setup complete.")


## 1. Download Data

Downloads the Higgs Twitter edgelists into `data/` (skips files that already exist).


In [ ]:
import importlib, dataloader
importlib.reload(dataloader)
dataloader.main()


## Helper Functions

Shared utilities: network loader, cached centrality computation, and reusable plot functions.


In [ ]:
# ── Network loader ────────────────────────────────────────────────────────────

def load_higgs_network(edgelist_name):
    """Load a Higgs edgelist (from data/) into a NetworkX DiGraph."""
    path = os.path.join(DATA_DIR, edgelist_name)
    with open(path) as f:
        return nx.read_edgelist(f, create_using=nx.DiGraph(), data=(('weight', int),))


# ── Centrality with JSON cache ─────────────────────────────────────────────────

def _load_or_compute(json_path, compute_fn):
    if os.path.exists(json_path):
        return json.load(open(json_path))
    print(f"  Computing (slow) → {json_path} …")
    result = compute_fn()
    json.dump(result, open(json_path, 'w'))
    return result

def load_or_compute_betweenness(G_ig, json_path):
    def _compute():
        return list(zip(G_ig.vs['_nx_name'], G_ig.betweenness()))
    return _load_or_compute(json_path, _compute)

def load_or_compute_eigenvector(G_ig, json_path):
    def _compute():
        return list(zip(G_ig.vs['_nx_name'], G_ig.eigenvector_centrality(directed=True)))
    return _load_or_compute(json_path, _compute)


# ── Degree distribution ────────────────────────────────────────────────────────

def plot_degree_distribution(in_deg, out_deg, title):
    """Log-log scatter of in- and out-degree distributions."""
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    fig.suptitle(f'{title}\nDegree Distribution', fontweight='bold')
    for ax, deg_list, label, color in [
        (axes[0], in_deg,  'In-Degree',  'cornflowerblue'),
        (axes[1], out_deg, 'Out-Degree', 'coral'),
    ]:
        cnt = Counter(d for _, d in deg_list)
        xs, ys = zip(*sorted(cnt.items()))
        ax.scatter(xs, ys, s=8, alpha=0.7, color=color)
        ax.set_xscale('log'); ax.set_yscale('log')
        ax.set_xlabel(label); ax.set_ylabel('Count')
        ax.set_title(f'{label} (log-log)')
        ax.grid(True, which='both', linestyle='--', alpha=0.4)
    plt.tight_layout()
    return fig


# ── Top-N centrality bar charts ────────────────────────────────────────────────

def plot_top_nodes(in_deg, out_deg, betweenness, eigenvector, title, n=15):
    """2×2 horizontal bar charts of top-N nodes for each centrality metric."""
    fig, axes = plt.subplots(2, 2, figsize=(14, 10))
    fig.suptitle(f'{title}\nTop {n} Nodes by Centrality Metric', fontweight='bold')
    datasets = [
        (in_deg[:n],      'In-Degree',             axes[0, 0], 'cornflowerblue'),
        (out_deg[:n],     'Out-Degree',             axes[0, 1], 'mediumseagreen'),
        (betweenness[:n], 'Betweenness Centrality', axes[1, 0], 'coral'),
        (eigenvector[:n], 'Eigenvector Centrality', axes[1, 1], 'mediumpurple'),
    ]
    for data, label, ax, color in datasets:
        nodes  = [str(x[0])[:20] for x in reversed(data)]
        values = [x[1]           for x in reversed(data)]
        ax.barh(nodes, values, color=color)
        ax.set_title(label); ax.set_xlabel('Value')
        ax.tick_params(axis='y', labelsize=7)
        ax.grid(axis='x', linestyle='--', alpha=0.4)
    plt.tight_layout()
    return fig


# ── Top-10% pairwise intersections ─────────────────────────────────────────────

def plot_intersections(in_deg, out_deg, betweenness, eigenvector, title):
    """Bar chart of pairwise top-10% metric intersections."""
    n = max(1, int(0.1 * len(in_deg)))
    sets = {
        'In-Deg':      set(x[0] for x in in_deg[:n]),
        'Out-Deg':     set(x[0] for x in out_deg[:n]),
        'Betweenness': set(x[0] for x in betweenness[:n]),
        'Eigenvector': set(x[0] for x in eigenvector[:n]),
    }
    pairs = [(f'{a} ∩ {b}', len(sets[a] & sets[b]))
             for i, a in enumerate(sets) for b in list(sets)[i+1:]]
    labels, counts = zip(*pairs)
    fig, ax = plt.subplots(figsize=(11, 4))
    bars = ax.bar(labels, counts, color='teal', edgecolor='black', alpha=0.8)
    ax.bar_label(bars, padding=3, fontsize=9)
    ax.set_title(f'{title}\nTop-10% Pairwise Metric Intersections')
    ax.set_ylabel('Node Count')
    ax.tick_params(axis='x', labelsize=8)
    ax.grid(axis='y', linestyle='--', alpha=0.4)
    plt.tight_layout()
    return fig


# ── Summary stats helper ───────────────────────────────────────────────────────

def summarize(name, G, in_deg, out_deg):
    avg_in  = sum(d for _, d in in_deg)  / len(in_deg)
    avg_out = sum(d for _, d in out_deg) / len(out_deg)
    print(f"[{name}]  nodes={G.order():,}  edges={G.size():,}"
          f"  avg_in={avg_in:.3f}  avg_out={avg_out:.3f}")

print("Helper functions defined.")


## 2. Higgs Mention Network


In [ ]:
print("=" * 55)
print("Higgs Mention Network")
print("=" * 55)

mention_G = load_higgs_network('higgs-mention_network.edgelist')

in_deg_mention  = sorted(mention_G.in_degree(),  key=lambda x: x[1], reverse=True)
out_deg_mention = sorted(mention_G.out_degree(), key=lambda x: x[1], reverse=True)
summarize('Mention', mention_G, in_deg_mention, out_deg_mention)

mention_ig = ig.Graph.from_networkx(mention_G)
bw_mention = sorted(
    load_or_compute_betweenness(mention_ig, 'mentions/mention_betweenness_centrality.json'),
    key=lambda x: x[1], reverse=True)
ev_mention = sorted(
    load_or_compute_eigenvector(mention_ig, 'mentions/mention_eigenvector_centrality.json'),
    key=lambda x: x[1], reverse=True)

fig = plot_degree_distribution(in_deg_mention, out_deg_mention, 'Higgs Mention Network')
all_figures.append(fig); savefig(fig, 'higgs_mention_deg.png'); plt.show()

fig = plot_top_nodes(in_deg_mention, out_deg_mention, bw_mention, ev_mention, 'Higgs Mention Network')
all_figures.append(fig); savefig(fig, 'higgs_mention_centrality.png'); plt.show()

fig = plot_intersections(in_deg_mention, out_deg_mention, bw_mention, ev_mention, 'Higgs Mention Network')
all_figures.append(fig); savefig(fig, 'higgs_mention_intersections.png'); plt.show()


## 3. Higgs Reply Network


In [ ]:
print("=" * 55)
print("Higgs Reply Network")
print("=" * 55)

reply_G = load_higgs_network('higgs-reply_network.edgelist')

in_deg_reply  = sorted(reply_G.in_degree(),  key=lambda x: x[1], reverse=True)
out_deg_reply = sorted(reply_G.out_degree(), key=lambda x: x[1], reverse=True)
summarize('Reply', reply_G, in_deg_reply, out_deg_reply)

reply_ig = ig.Graph.from_networkx(reply_G)
bw_reply = sorted(
    load_or_compute_betweenness(reply_ig, 'Reply/reply_betweenness_centrality.json'),
    key=lambda x: x[1], reverse=True)
ev_reply = sorted(
    load_or_compute_eigenvector(reply_ig, 'Reply/reply_eigenvector_centrality.json'),
    key=lambda x: x[1], reverse=True)

fig = plot_degree_distribution(in_deg_reply, out_deg_reply, 'Higgs Reply Network')
all_figures.append(fig); savefig(fig, 'higgs_reply_deg.png'); plt.show()

fig = plot_top_nodes(in_deg_reply, out_deg_reply, bw_reply, ev_reply, 'Higgs Reply Network')
all_figures.append(fig); savefig(fig, 'higgs_reply_centrality.png'); plt.show()

fig = plot_intersections(in_deg_reply, out_deg_reply, bw_reply, ev_reply, 'Higgs Reply Network')
all_figures.append(fig); savefig(fig, 'higgs_reply_intersections.png'); plt.show()


## 4. Higgs Retweet Network


In [ ]:
print("=" * 55)
print("Higgs Retweet Network")
print("=" * 55)

retweet_G = load_higgs_network('higgs-retweet_network.edgelist')

in_deg_retweet  = sorted(retweet_G.in_degree(),  key=lambda x: x[1], reverse=True)
out_deg_retweet = sorted(retweet_G.out_degree(), key=lambda x: x[1], reverse=True)
summarize('Retweet', retweet_G, in_deg_retweet, out_deg_retweet)

retweet_ig = ig.Graph.from_networkx(retweet_G)
bw_retweet = sorted(
    load_or_compute_betweenness(retweet_ig, 'Retweet/retweet_betweenness_centrality.json'),
    key=lambda x: x[1], reverse=True)
ev_retweet = sorted(
    load_or_compute_eigenvector(retweet_ig, 'Retweet/retweet_eigenvector_centrality.json'),
    key=lambda x: x[1], reverse=True)

fig = plot_degree_distribution(in_deg_retweet, out_deg_retweet, 'Higgs Retweet Network')
all_figures.append(fig); savefig(fig, 'higgs_retweet_deg.png'); plt.show()

fig = plot_top_nodes(in_deg_retweet, out_deg_retweet, bw_retweet, ev_retweet, 'Higgs Retweet Network')
all_figures.append(fig); savefig(fig, 'higgs_retweet_centrality.png'); plt.show()

fig = plot_intersections(in_deg_retweet, out_deg_retweet, bw_retweet, ev_retweet, 'Higgs Retweet Network')
all_figures.append(fig); savefig(fig, 'higgs_retweet_intersections.png'); plt.show()


## 5. Cross-Network Comparison (Higgs)


In [ ]:
print("=" * 55)
print("Cross-Network Comparison")
print("=" * 55)

network_names = ['Mention', 'Reply', 'Retweet']
graphs        = [mention_G, reply_G, retweet_G]
in_deg_lists  = [in_deg_mention, in_deg_reply, in_deg_retweet]

node_counts = [G.order() for G in graphs]
edge_counts = [G.size()  for G in graphs]
avg_in_degs = [sum(d for _, d in dl) / len(dl) for dl in in_deg_lists]

fig, axes = plt.subplots(1, 3, figsize=(15, 5))
fig.suptitle('Higgs Twitter Networks — Side-by-Side Comparison', fontweight='bold')

for ax, vals, title, color, fmt in [
    (axes[0], node_counts, 'Nodes',         'cornflowerblue', '%,.0f'),
    (axes[1], edge_counts, 'Edges',         'coral',          '%,.0f'),
    (axes[2], avg_in_degs, 'Avg In-Degree', 'mediumseagreen', '%.3f'),
]:
    bars = ax.bar(network_names, vals, color=color, edgecolor='black')
    ax.bar_label(bars, fmt=fmt, padding=3, fontsize=9)
    ax.set_title(title); ax.set_ylabel(title)
    ax.grid(axis='y', linestyle='--', alpha=0.4)

plt.tight_layout()
all_figures.append(fig); savefig(fig, 'higgs_comparison.png'); plt.show()

# Cross-network top-10% in-degree overlap
def top10_set(dl):
    n = max(1, int(0.1 * len(dl)))
    return set(x[0] for x in dl[:n])

in_sets = [top10_set(dl) for dl in in_deg_lists]

cross = {
    'Mention ∩ Reply':   len(in_sets[0] & in_sets[1]),
    'Mention ∩ Retweet': len(in_sets[0] & in_sets[2]),
    'Reply ∩ Retweet':   len(in_sets[1] & in_sets[2]),
    'All Three':          len(in_sets[0] & in_sets[1] & in_sets[2]),
}
fig, ax = plt.subplots(figsize=(9, 4))
bars = ax.bar(list(cross.keys()), list(cross.values()), color='teal', edgecolor='black', alpha=0.8)
ax.bar_label(bars, padding=3)
ax.set_title('Cross-Network Top-10% In-Degree Overlap')
ax.set_ylabel('Node Count')
ax.grid(axis='y', linestyle='--', alpha=0.4)
plt.tight_layout()
all_figures.append(fig); savefig(fig, 'higgs_overlap.png'); plt.show()

# Combined 3×2 degree distribution panel for the paper
fig, axes = plt.subplots(3, 2, figsize=(12, 12))
fig.suptitle('Higgs Twitter Networks — Degree Distributions (log-log)', fontweight='bold', fontsize=13)
for row, (net_name, in_d, out_d) in enumerate([
    ('Mention',  in_deg_mention,  out_deg_mention),
    ('Reply',    in_deg_reply,    out_deg_reply),
    ('Retweet',  in_deg_retweet,  out_deg_retweet),
]):
    for col, (deg_list, label, color) in enumerate([
        (in_d,  'In-Degree',  'cornflowerblue'),
        (out_d, 'Out-Degree', 'coral'),
    ]):
        ax = axes[row, col]
        cnt = Counter(d for _, d in deg_list)
        xs, ys = zip(*sorted(cnt.items()))
        ax.scatter(xs, ys, s=8, alpha=0.7, color=color)
        ax.set_xscale('log'); ax.set_yscale('log')
        ax.set_xlabel(label); ax.set_ylabel('Count')
        ax.set_title(f'{net_name} — {label}')
        ax.grid(True, which='both', linestyle='--', alpha=0.4)
plt.tight_layout()
all_figures.append(fig); savefig(fig, 'higgs_all_deg.png'); plt.show()


## 6. Enron Email Network

Uses the pre-built edgelist and cached centrality JSONs from `enron_full_analysis_final/`.
Community data loaded from `top_nodes_per_community.json` (Louvain, pre-computed).


In [ ]:
print("=" * 55)
print("Enron Email Network")
print("=" * 55)

ENRON_DIR = os.path.join(ROOT, 'enron_full_analysis_final')

enron_G = nx.read_edgelist(
    os.path.join(ENRON_DIR, 'enron_network.edgelist'),
    delimiter='\t', create_using=nx.DiGraph())
print(f"Nodes: {enron_G.order():,}   Edges: {enron_G.size():,}")

in_deg_enron  = sorted(enron_G.in_degree(),  key=lambda x: x[1], reverse=True)
out_deg_enron = sorted(enron_G.out_degree(), key=lambda x: x[1], reverse=True)

bw_enron = sorted(
    json.load(open(os.path.join(ENRON_DIR, 'enron_betweenness_centrality.json'))),
    key=lambda x: x[1], reverse=True)
ev_enron = sorted(
    json.load(open(os.path.join(ENRON_DIR, 'enron_eigenvector_centrality.json'))),
    key=lambda x: x[1], reverse=True)

# Degree distribution (histogram, log scale)
enron_un = enron_G.to_undirected()
degrees  = [d for _, d in enron_un.degree()]
fig, ax  = plt.subplots(figsize=(8, 5))
ax.hist(degrees, bins=50, log=True, color='steelblue', edgecolor='black', alpha=0.8)
ax.set_title('Enron Email Network\nDegree Distribution (Log Scale)')
ax.set_xlabel('Degree'); ax.set_ylabel('Frequency (log)')
ax.grid(axis='y', linestyle='--', alpha=0.4)
plt.tight_layout()
all_figures.append(fig); savefig(fig, 'enron_deg.png'); plt.show()

# Top nodes (note: node labels are email addresses — truncated to 20 chars)
fig = plot_top_nodes(in_deg_enron, out_deg_enron, bw_enron, ev_enron, 'Enron Email Network')
all_figures.append(fig); savefig(fig, 'enron_centrality.png'); plt.show()

# Community sizes from pre-computed JSON
community_data  = json.load(open(os.path.join(ENRON_DIR, 'top_nodes_per_community (1).json')))
comm_sizes_enron = sorted([c['size'] for c in community_data], reverse=True)
print(f"Communities: {len(comm_sizes_enron)}")

top_n = 30
fig, ax = plt.subplots(figsize=(10, 4))
ax.bar(range(top_n), comm_sizes_enron[:top_n], color='coral', edgecolor='black', alpha=0.8)
for i, sz in enumerate(comm_sizes_enron[:top_n]):
    ax.text(i, sz + 5, str(sz), ha='center', va='bottom', fontsize=7, rotation=45)
ax.set_title(f'Enron Email Network\nTop {top_n} Community Sizes (of {len(comm_sizes_enron)} total, Louvain)')
ax.set_xlabel(f'Community rank (1–{top_n}, sorted by size)'); ax.set_ylabel('Nodes')
ax.grid(axis='y', linestyle='--', alpha=0.4)
plt.tight_layout()
all_figures.append(fig); savefig(fig, 'enron_communities.png'); plt.show()


## 7. Facebook Combined Network


In [ ]:
print("=" * 55)
print("Facebook Combined Network")
print("=" * 55)

FB_DIR = os.path.join(ROOT, 'facebook_combined')
fb_G   = nx.read_edgelist(
    os.path.join(FB_DIR, 'facebook_combined.txt'),
    create_using=nx.Graph(), nodetype=int)
print(f"Nodes: {fb_G.order():,}   Edges: {fb_G.size():,}")

# Degree distribution
fb_degrees = [d for _, d in fb_G.degree()]
fig, ax    = plt.subplots(figsize=(8, 5))
ax.hist(fb_degrees, bins=50, log=True, color='mediumseagreen', edgecolor='black', alpha=0.8)
ax.set_title('Facebook Network\nDegree Distribution (Log Scale)')
ax.set_xlabel('Degree'); ax.set_ylabel('Frequency (log)')
ax.grid(axis='y', linestyle='--', alpha=0.4)
plt.tight_layout()
all_figures.append(fig); savefig(fig, 'fb_deg.png'); plt.show()

# Louvain community detection
print("Running Louvain community detection …")
fb_communities  = nx.community.louvain_communities(fb_G, seed=42)
fb_comm_sizes   = sorted([len(c) for c in fb_communities], reverse=True)
print(f"Communities: {len(fb_communities)}  sizes: {fb_comm_sizes}")

fig, ax = plt.subplots(figsize=(12, 4))
ax.bar(range(len(fb_comm_sizes)), fb_comm_sizes,
       color='mediumpurple', edgecolor='black', alpha=0.8)
for i, sz in enumerate(fb_comm_sizes):
    ax.text(i, sz + 3, str(sz), ha='center', va='bottom', fontsize=8)
ax.set_title(f'Facebook Network\nLouvain Community Sizes ({len(fb_communities)} communities)')
ax.set_xlabel('Community (sorted by size)'); ax.set_ylabel('Nodes')
ax.grid(axis='y', linestyle='--', alpha=0.4)
plt.tight_layout()
all_figures.append(fig); savefig(fig, 'fb_communities.png'); plt.show()

# Hub degree per community
degree_dict = dict(fb_G.degree())
sorted_comms = sorted(fb_communities, key=len, reverse=True)
hub_degs = [max(degree_dict[n] for n in c) for c in sorted_comms]

fig, ax = plt.subplots(figsize=(12, 4))
ax.bar(range(len(hub_degs)), hub_degs, color='mediumpurple', edgecolor='black', alpha=0.6)
ax.set_title('Facebook Network\nHub Node Degree per Community (Largest Communities First)')
ax.set_xlabel('Community'); ax.set_ylabel('Max Degree in Community')
ax.grid(axis='y', linestyle='--', alpha=0.4)
plt.tight_layout()
all_figures.append(fig); savefig(fig, 'fb_hub_degrees.png'); plt.show()


## 8. Export to PDF

Compiles every figure generated above into a single multi-page PDF.


In [ ]:
import datetime

pdf_path = os.path.join(ROOT, 'network_analysis_results.pdf')

with PdfPages(pdf_path) as pdf:
    # ── Title page ──
    tfig = plt.figure(figsize=(11, 8.5))
    tfig.patch.set_facecolor('#f8f8f8')
    tfig.text(0.5, 0.62, 'ECE 227 — Network Analysis Results',
              ha='center', va='center', fontsize=26, fontweight='bold')
    tfig.text(0.5, 0.50,
              'Higgs Twitter (Mention · Reply · Retweet)\nEnron Email · Facebook Networks',
              ha='center', va='center', fontsize=15, color='#444444', linespacing=1.8)
    tfig.text(0.5, 0.36, f'Generated: {datetime.date.today()}',
              ha='center', va='center', fontsize=11, color='#888888')
    pdf.savefig(tfig, bbox_inches='tight')
    plt.close(tfig)

    for fig in all_figures:
        pdf.savefig(fig, bbox_inches='tight')
        plt.close(fig)

total_pages = 1 + len(all_figures)
print(f"Saved {total_pages} pages to: {pdf_path}")
